## runs on Colab

In [1]:
!pip install -q fastapi uvicorn pyngrok transformers torch

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:
!pip install -U -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 83.0 MB/s eta 0:00:00


In [4]:
from transformers import BartForConditionalGeneration, BartTokenizer

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

model = model.to("cuda")

print("Model loaded successfully")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully


In [5]:
!pip install -q fastapi uvicorn pyngrok transformers torch

In [6]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class SummarizeRequest(BaseModel):
    text: str

@app.post("/summarize")
def summarize(request: SummarizeRequest):

    inputs = tokenizer(
        request.text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    summary_ids = model.generate(
        **inputs,
        max_length=150,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return {
        "summary": summary
    }

In [7]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.start()

In [9]:
import requests

response = requests.get("http://127.0.0.1:8000/docs")

print(response.status_code)

INFO:     127.0.0.1:47064 - "GET /docs HTTP/1.1" 200 OK
200


In [ ]:
from pyngrok import ngrok

ngrok_token = ""
ngrok.set_auth_token("ngrok_token")

In [11]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://scalding-creme-broiling.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
# Test
text = """
Artificial intelligence is transforming many industries.
Machine learning allows computers to learn patterns from data.
Deep learning uses neural networks with multiple layers
to solve complex problems such as image recognition,
natural language processing, and speech recognition.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

inputs = {k: v.to("cuda") for k, v in inputs.items()}

summary_ids = model.generate(
    **inputs,
    max_length=150,
    min_length=30,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

Machine learning allows computers to learn patterns from data. Deep learning uses neural networks with multiple layers to solve complex problems such as image recognition,natural language processing, and speech recognition.
